<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/O_Gram_Maitre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy as np
from itertools import product, combinations_with_replacement
from math import factorial
from collections import Counter
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import roc_auc_score
from sklearn.datasets import load_digits
from sklearn.svm import SVC
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV
from scipy.optimize import minimize, LinearConstraint

# ══════════════════════════════════════════════════════════════════════════════
# PARAMÈTRES
# ══════════════════════════════════════════════════════════════════════════════
params = {
    "neg_digits":  [3, 5],
    "pos_digits":  [8],
    "img_size":    4,          # 8×8 -> 4×4  (dim ambiante 16)
    "n_train":     40,

    "n_unlabeled": 4000,
    "seed":        12345,
    "seeds":       {42: 73559, 43: 9658},

    "n_test":      500,
    "weights":        {0: 0, 1: 1.0, 2: 0.5, 3: 0.15},   # poids QP complet
    "weights_screen": {0: 0, 1: 1.0},                   # poids screening (moins cher)
    "lambda_G":       1e-9,
    "qp_margin":      1.0,
    "thres1":         1e-4,
    "thres2":         1e4,
    "const_pen":      1e-4,
    "n_G":            600,      # taille de l'échantillon cloud pour les moments
    "batch_size":     200,      # taille des blocs de points du nuage (streaming)

    "deg_start":   3,           # degré ambiant de départ
    "new_deg":     4,           # degré des candidats ajoutés — RESTE FIXE
    "n_rounds":    2,           # nombre de tours (blocs successifs) à ce degré
    "dict_size":   4000,        # taille du dictionnaire de candidats à chaque tour
    "n_select":    1000,         # nombre retenu après screening H, à chaque tour
}





# ══════════════════════════════════════════════════════════════════════════════
# DONNÉES : sklearn digits, {3,5} vs {8}, réduits 8×8 -> 4×4
# ══════════════════════════════════════════════════════════════════════════════
from tensorflow.keras.datasets import mnist

def load_mnist_binary(neg_list, pos_list, img_size,
                      n_train, n_test, n_unlabeled, seed=params["seeds"][42]):

    (Xtr, ytr), (Xte, yte) = mnist.load_data()

    X64 = np.concatenate([Xtr, Xte], axis=0).astype(float)
    y_raw = np.concatenate([ytr, yte], axis=0)

    keep = set(neg_list) | set(pos_list)
    mask = np.isin(y_raw, list(keep))
    X64 = X64[mask]
    y_raw = y_raw[mask]

    y = np.where(np.isin(y_raw, list(pos_list)), 1.0, -1.0)

    # réduction 28×28 -> img_size×img_size
    if 28 % img_size != 0:
        raise ValueError("img_size doit diviser 28")

    b = 28 // img_size
    Xs = X64.reshape(-1, img_size, b, img_size, b).mean(axis=(2,4))
    Xs = Xs.reshape(-1, img_size * img_size)

    Xs = (Xs - Xs.mean(0)) / (Xs.std(0) + 1e-8)

    rng = np.random.default_rng(seed)

    idx_neg = np.where(y < 0)[0]
    idx_pos = np.where(y > 0)[0]
    rng.shuffle(idx_neg)
    rng.shuffle(idx_pos)

    n_tr_c = n_train // 2
    n_te_c = n_test // 2

    tr = np.concatenate([idx_neg[:n_tr_c], idx_pos[:n_tr_c]])
    te = np.concatenate([idx_neg[n_tr_c:n_tr_c+n_te_c],
                         idx_pos[n_tr_c:n_tr_c+n_te_c]])

    used = set(tr) | set(te)
    unlab_pool = np.array([i for i in range(len(Xs)) if i not in used])

    if len(unlab_pool) > n_unlabeled:
        unlab = rng.choice(unlab_pool, n_unlabeled, replace=False)
    else:
        unlab = unlab_pool

    rng.shuffle(tr)
    rng.shuffle(te)
    rng.shuffle(unlab)

    return Xs[tr], y[tr], Xs[te], y[te], Xs[unlab], y[unlab]


# ══════════════════════════════════════════════════════════════════════════════
# QP SOBOLEV — solveur générique
# ══════════════════════════════════════════════════════════════════════════════
def solve_qp(A, y, G, margin):
    n, k = A.shape
    Gr = G + 1e-12 * np.eye(k)
    con = LinearConstraint(np.diag(y) @ A, lb=margin, ub=np.inf)
    try:    c0 = np.linalg.lstsq(A, 1.5 * margin * y, rcond=None)[0]
    except Exception: c0 = np.zeros(k)
    res = minimize(lambda c: c @ Gr @ c, c0, jac=lambda c: 2 * Gr @ c,
                   constraints=[con], method='SLSQP',
                   options={'maxiter': 500, 'ftol': 1e-11})
    c = res.x
    marge_eff = float(np.min(y * (A @ c)))
    return c, marge_eff >= margin - 1e-4, marge_eff


# ══════════════════════════════════════════════════════════════════════════════
# BASELINE AMBIANTE SIMPLE (référence historique, PolynomialFeatures standard)
# ══════════════════════════════════════════════════════════════════════════════
def classify_poly_qp(X_tr, y_tr, X_te, y_te, X_cloud, deg, weights,
                     thres1, thres2, const_pen, qp_margin, n_G, titre=""):
    n, dd = X_tr.shape
    lo = X_cloud.min(axis=0); hi = X_cloud.max(axis=0)
    span = np.where(hi - lo > 1e-12, hi - lo, 1.0)
    def to_u(X): return 2 * (X - lo) / span - 1.0
    SC = 2.0 / span

    poly = PolynomialFeatures(degree=deg, include_bias=False)
    Phi_tr = poly.fit_transform(to_u(X_tr))
    powers = poly.powers_; n_feat = Phi_tr.shape[1]

    def pderiv(U, coefs, dims=()):
        P = powers.astype(float).copy(); m = coefs.astype(float).copy()
        for dm in dims: m = m * P[:, dm]; P[:, dm] -= 1
        v = m != 0
        return np.zeros(len(U)) if not v.any() else (U[:, None, :] ** P[None, v, :]).prod(2) @ m[v]

    rng = np.random.default_rng(params["seeds"][43])
    idx = rng.choice(len(X_cloud), min(n_G, len(X_cloud)), replace=False)
    U_G = to_u(X_cloud[idx]); nG = len(idx)
    PHI = poly.transform(U_G)
    E = np.eye(n_feat)
    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.)
    G = np.zeros((n_feat, n_feat))
    if w0: G += w0 * (PHI.T @ PHI) / nG
    if w1:
        GG = np.zeros((nG, n_feat, dd))
        for j in range(n_feat):
            for a in range(dd):
                if powers[j, a] > 0:
                    GG[:, j, a] = SC[a] * pderiv(U_G, E[j], (a,))
        G += w1 * np.einsum('xik,xjk->ij', GG, GG) / nG
    G += 1e-9 * np.eye(n_feat)

    mean_phi = PHI.mean(axis=0)
    s_g, V_g = np.linalg.eigh(G)
    keep = (s_g > thres1) & (s_g < thres2)
    if keep.sum() == 0:
        raise ValueError("bande spectrale vide")
    T = V_g[:, keep] / np.sqrt(s_g[keep]); r = int(keep.sum())
    A_qp = np.hstack([(Phi_tr - mean_phi) @ T, np.ones((n, 1))])
    G_qp = np.eye(r + 1); G_qp[-1, -1] = const_pen
    sol, feas, marge = solve_qp(A_qp, y_tr, G_qp, qp_margin)
    coef = T @ sol[:r]; off = sol[-1] - mean_phi @ coef

    f_te = poly.transform(to_u(X_te)) @ coef + off
    f_tr = Phi_tr @ coef + off
    mse_te = float(np.mean((f_te - y_te) ** 2))
    normH = float(np.sqrt(max(sol[:r] @ sol[:r], 0)))
    def acc(f, y):
        sg = np.sign(f)
        return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
    # seuil recentré : demi-somme des moyennes de f sur les deux classes (train)
    thr = 0.5 * (f_tr[y_tr > 0].mean() + f_tr[y_tr < 0].mean())
    def acc_thr(f, y, t):
        sg = np.sign(f - t)
        return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
    acc_te_thr = acc_thr(f_te, y_te, thr)
    try:    auc = roc_auc_score((y_te > 0).astype(int), f_te)
    except Exception: auc = float('nan')
    print(f"  [{titre}] deg={deg} feat={n_feat} rang={r} | faisable={feas} marge={marge:.3f} "
          f"‖u‖_H={normH:.4f} MSE_te={mse_te:.4f} | "
          f"acc_tr={acc(f_tr,y_tr):.3f} acc_te={acc(f_te,y_te):.4f} "
          f"acc_te_seuil={acc_te_thr:.4f} (thr={thr:.4f}) AUC={auc:.4f}")
    return {'acc_te': acc(f_te, y_te), 'acc_te_thr': acc_te_thr, 'thr': thr,
            'auc': auc, 'mse_te': mse_te, 'normH': normH}

def classify_ridge(X_tr, y_tr, X_te, y_te, deg, titre=""):
    poly = PolynomialFeatures(degree=deg)
    r = Ridge(alpha=1e-8).fit(poly.fit_transform(X_tr), y_tr)
    f = r.predict(poly.transform(X_te))
    a = np.mean(np.sign(f) == np.sign(y_te))
    mse = float(np.mean((f - y_te) ** 2))
    try:    auc = roc_auc_score((y_te > 0).astype(int), f)
    except Exception: auc = float('nan')
    print(f"  [{titre}] Ridge deg={deg} : acc_te={a:.4f} MSE_te={mse:.4f} AUC={auc:.4f}")
    return {'acc_te': a, 'auc': auc, 'mse_te': mse}


# ══════════════════════════════════════════════════════════════════════════════
# ARCHITECTURE "POOL DE MOMENTS" — TOUT (Gram QP, corrélation H de sélection,
# NORMES des candidats du dictionnaire) est dérivé du MÊME pool, calculé une
# seule fois par appel. Aucune norme n'est recalculée "à part".
#
#   - une dérivée de monôme EST un monôme (à un coefficient scalaire près) :
#     réduction algébrique de l'exposant + cache d'évaluation par exposant
#     DISTINCT (MonomialCache) — pas de recalcul redondant ;
#   - pas de tenseur dense (dd,dd,dd) : boucle sur les combos DISTINCTS de dims
#     à multiplicité près (C(dd+o-1,o), petit même en dimension ambiante pleine) ;
#   - validé à 1e-15/1e-16 près contre les calculs directs (voir historique).
# ══════════════════════════════════════════════════════════════════════════════
def _multiplicity(combo):
    c = Counter(combo); m = factorial(len(combo))
    for v in c.values(): m //= factorial(v)
    return m

def poly_eval_from_powers(U, powers):
    n = U.shape[0]; nf = powers.shape[0]
    Phi = np.ones((n, nf))
    for a in range(U.shape[1]):
        e = powers[:, a]
        if not np.any(e): continue
        maxe = int(e.max())
        pw = np.empty((maxe + 1, n)); pw[0] = 1.0
        if maxe >= 1: pw[1] = U[:, a]
        for p in range(2, maxe + 1): pw[p] = pw[p - 1] * U[:, a]
        Phi *= pw[e].T
    return Phi

class MonomialCache:
    """Cache des évaluations U^alpha (vecteur (nG,)) par tuple d'exposants."""
    def __init__(self, U_G):
        self.U_G = U_G
        self.cache = {}
    def eval(self, alpha):
        key = tuple(int(x) for x in alpha)
        if key not in self.cache:
            v = np.ones(self.U_G.shape[0])
            for d, e in enumerate(key):
                if e: v = v * self.U_G[:, d] ** e
            self.cache[key] = v
        return self.cache[key]

def _reduce_monomial(p, combo_count):
    coeff = 1.0
    r = p.copy()
    for a, k in combo_count.items():
        if r[a] < k:
            return 0.0, None
        for i in range(k):
            coeff *= (r[a] - i)
        r[a] -= k
    return coeff, r

# ══════════════════════════════════════════════════════════════════════════════
# GRAM MAÎTRE — calculé UNE SEULE FOIS pour tout le pool (actifs de départ ∪
# TOUT le dictionnaire de degré new_deg), par ordre de dérivation. Ensuite,
# plus AUCUN accès au nuage : Gram QP, corrélation H de sélection, normes,
# centrage — tout est un SLICING (np.ix_) + somme pondérée du Gram maître.
#
#   - économie automatique : master[o] n'est calculé QUE si weights[o] ou
#     weights_screen[o] est non nul. Si w0=0, l'ordre 0 (qui exigerait le
#     degré complet 2·new_deg) n'est simplement jamais construit — seuls les
#     ordres >=1 le sont, dont le degré de moment nécessaire est <= 2·new_deg-2 ;
#   - dérivée de monôme = monôme (coefficient scalaire + exposant réduit) :
#     réduction algébrique + MonomialCache par exposant distinct ;
#   - double streaming (combos de dims, ET blocs de points du nuage) pendant
#     la construction du maître — mémoire de crête indépendante de N et de nG ;
#   - validé à 1e-15 près contre le calcul direct (voir historique).
# ══════════════════════════════════════════════════════════════════════════════
def build_master_grams(U_G, powers_master, orders_needed, SC, nG, batch_size=200):
    """Gram COMPLET (N×N) par ordre, NON pondéré par w_o (juste Σ_combo mult *
    produit). N = len(powers_master). Une seule passe sur le nuage, batchée
    sur les points ET sur les combos. Retourne (master, mean0) où mean0 est
    la moyenne de PHI (ordre 0) sur tout le pool, pour le centrage exact."""
    N = powers_master.shape[0]; dd = U_G.shape[1]
    master = {o: np.zeros((N, N)) for o in orders_needed}
    mean0 = np.zeros(N) if 0 in orders_needed else None

    n_batches = int(np.ceil(nG / batch_size))
    for b in range(n_batches):
        U_b = U_G[b * batch_size:(b + 1) * batch_size]
        nb = U_b.shape[0]
        mc = MonomialCache(U_b)
        if 0 in orders_needed:
            PHI = np.zeros((nb, N))
            for j in range(N):
                PHI[:, j] = mc.eval(powers_master[j])
            master[0] += PHI.T @ PHI
            mean0 += PHI.sum(axis=0)
            del PHI
        for o in [oo for oo in orders_needed if oo >= 1]:
            for combo in combinations_with_replacement(range(dd), o):
                mult = _multiplicity(combo)
                combo_count = Counter(combo)
                sc_factor = np.prod([SC[a] for a in combo])
                GG = np.zeros((nb, N))
                for j in range(N):
                    coeff, alpha = _reduce_monomial(powers_master[j], combo_count)
                    if coeff:
                        GG[:, j] = sc_factor * coeff * mc.eval(alpha)
                master[o] += mult * (GG.T @ GG)
                del GG
    for o in master:
        master[o] /= nG
    if mean0 is not None:
        mean0 /= nG
    return master, mean0

def gram_from_master(master, weights, idx_rows, idx_cols):
    """PUR slicing + somme pondérée — zéro accès au nuage."""
    G = np.zeros((len(idx_rows), len(idx_cols)))
    for o, wo in weights.items():
        if wo and o in master:
            G += wo * master[o][np.ix_(idx_rows, idx_cols)]
    return G

def diag_from_master(master, weights, idx):
    """Diagonale (normes ‖.‖²_H) — pur slicing (arr[idx, idx])."""
    idx_arr = np.asarray(idx, dtype=int)   # dtype forcé : liste vide -> float par défaut sinon
    d = np.zeros(len(idx))
    for o, wo in weights.items():
        if wo and o in master:
            d += wo * master[o][idx_arr, idx_arr]
    return d


# ══════════════════════════════════════════════════════════════════════════════
# QP sur un sous-ensemble d'indices du pool maître — PUR algèbre linéaire
# ══════════════════════════════════════════════════════════════════════════════
def fit_qp_from_master(X_tr, y_tr, X_te, y_te, powers_master, idx, lo, span,
                       master, mean0, weights, lambda_G, thres1, thres2,
                       const_pen, qp_margin, titre=""):
    n = X_tr.shape[0]
    powers = powers_master[idx]
    def to_u(X): return 2 * (X - lo) / span - 1.0
    Phi_tr = poly_eval_from_powers(to_u(X_tr), powers)

    G = gram_from_master(master, weights, idx, idx) + lambda_G * np.eye(len(idx))
    mean_phi = mean0[idx] if mean0 is not None else np.zeros(len(idx))

    s_g, V_g = np.linalg.eigh(G)
    keep = (s_g > thres1) & (s_g < thres2)
    if keep.sum() == 0:
        raise ValueError("bande spectrale vide")
    T = V_g[:, keep] / np.sqrt(s_g[keep]); r = int(keep.sum())
    A_qp = np.hstack([(Phi_tr - mean_phi) @ T, np.ones((n, 1))])
    G_qp = np.eye(r + 1); G_qp[-1, -1] = const_pen
    sol, feas, marge = solve_qp(A_qp, y_tr, G_qp, qp_margin)
    coef = T @ sol[:r]; off = sol[-1] - mean_phi @ coef

    f_te = poly_eval_from_powers(to_u(X_te), powers) @ coef + off
    f_tr = Phi_tr @ coef + off
    mse_te = float(np.mean((f_te - y_te) ** 2))
    normH = float(np.sqrt(max(sol[:r] @ sol[:r], 0)))
    def acc(f, y):
        sg = np.sign(f)
        return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
    thr = 0.5 * (f_tr[y_tr > 0].mean() + f_tr[y_tr < 0].mean())
    def acc_thr(f, y, t):
        sg = np.sign(f - t)
        return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
    acc_te_thr = acc_thr(f_te, y_te, thr)
    try:    auc = roc_auc_score((y_te > 0).astype(int), f_te)
    except Exception: auc = float('nan')
    print(f"  [{titre}] n_feat={len(idx)} rang={r} | faisable={feas} marge={marge:.3f} "
          f"‖u‖_H={normH:.4f} MSE_te={mse_te:.4f} | "
          f"acc_tr={acc(f_tr,y_tr):.3f} acc_te={acc(f_te,y_te):.4f} "
          f"acc_te_seuil={acc_te_thr:.4f} (thr={thr:.4f}) AUC={auc:.4f}")
    return {'coef': coef, 'off': off, 'mse_te': mse_te, 'normH': normH,
            'acc_te': acc(f_te, y_te), 'acc_te_thr': acc_te_thr, 'thr': thr, 'auc': auc}


# ══════════════════════════════════════════════════════════════════════════════
# ÉTAPE GLOUTONNE — PUR slicing du Gram maître, aucun accès au nuage
# ══════════════════════════════════════════════════════════════════════════════
def greedy_add_degree_master(X_train, y_train, X_test, y_test, powers_master,
                             idx_active, idx_cand_all, prev_coef, prev_normH,
                             master, mean0, offset, dict_size, n_select,
                             weights, weights_screen, lambda_G,
                             thres1, thres2, const_pen, qp_margin, lo, span, titre_extra=""):
    idx_block = idx_cand_all[offset:offset + dict_size]
    print(f"\n── bloc [{offset}:{offset+len(idx_block)}] : "
          f"{len(idx_block)} monômes (sur {len(idx_cand_all)} candidats au total)")

    Gc = gram_from_master(master, weights_screen, idx_block, idx_active)
    proj = Gc @ prev_coef
    diag_cand = diag_from_master(master, weights_screen, idx_block)
    cos_sim = proj / (np.sqrt(np.maximum(diag_cand, 1e-30)) * max(prev_normH, 1e-30))

    order_sel = np.argsort(-np.abs(cos_sim))[:n_select]
    print(f"   sélection : {len(order_sel)}/{len(idx_block)} monômes "
          f"(|cos_H| min retenu = {np.abs(cos_sim[order_sel]).min():.4f}, "
          f"max = {np.abs(cos_sim[order_sel]).max():.4f})")
    print(f"   normes ‖φ‖_H des candidats sélectionnés : "
          f"min={np.sqrt(diag_cand[order_sel]).min():.4f} "
          f"max={np.sqrt(diag_cand[order_sel]).max():.4f}")

    idx_new_active = idx_active + [idx_block[i] for i in order_sel]

    sol = fit_qp_from_master(X_train, y_train, X_test, y_test, powers_master,
                             idx_new_active, lo, span, master, mean0, weights,
                             lambda_G, thres1, thres2, const_pen, qp_margin,
                             titre=f"QP actifs({len(idx_active)}) + {n_select} monômes "
                                   f"{titre_extra}(bloc offset={offset})")
    return idx_new_active, sol


def run_greedy_fixed_degree_master(X_train, y_train, X_test, y_test, powers_master,
                                   idx_active_start, idx_cand_all, sol_start,
                                   n_rounds, dict_size, n_select,
                                   master, mean0, weights, weights_screen, lambda_G,
                                   thres1, thres2, const_pen, qp_margin, lo, span):
    """Enchaîne n_rounds étapes gloutonnes — TOUT vient du Gram maître déjà
    calculé, plus aucun accès au nuage à partir d'ici."""
    idx_active = list(idx_active_start)
    coef, normH = sol_start['coef'], sol_start['normH']
    history = [{
        'round': 0, 'n_feat': len(idx_active),
        'mse_te': sol_start['mse_te'], 'normH': normH,
        'acc_te': sol_start['acc_te'],
        'acc_te_thr': sol_start.get('acc_te_thr', float('nan')),
        'thr': sol_start.get('thr', float('nan')),
        'auc': sol_start['auc'],
    }]
    for k in range(n_rounds):
        offset = k * dict_size
        if offset >= len(idx_cand_all):
            print(f"\n>> dictionnaire de candidats épuisé ({len(idx_cand_all)} au total, "
                  f"offset={offset}) : arrêt anticipé après {k}/{n_rounds} tours")
            break
        idx_active, sol = greedy_add_degree_master(
            X_train, y_train, X_test, y_test, powers_master,
            idx_active, idx_cand_all, coef, normH, master, mean0,
            offset, dict_size, n_select, weights, weights_screen, lambda_G,
            thres1, thres2, const_pen, qp_margin, lo, span)
        coef, normH = sol['coef'], sol['normH']
        history.append({
            'round': k + 1, 'n_feat': len(idx_active),
            'mse_te': sol['mse_te'], 'normH': normH,
            'acc_te': sol['acc_te'],
            'acc_te_thr': sol.get('acc_te_thr', float('nan')),
            'thr': sol.get('thr', float('nan')),
            'auc': sol['auc'],
        })
    print("\n" + "=" * 72)
    print(f"HISTORIQUE GLOUTON (Gram maître, par tour)")
    print("=" * 72)
    for h in history:
        print(f"  tour={h['round']:<2d} n_feat={h['n_feat']:<5d} "
              f"MSE_te={h['mse_te']:.4f}  ‖u‖_H={h['normH']:.4f}  "
              f"acc_te={h['acc_te']:.4f}  acc_te_seuil={h['acc_te_thr']:.4f}  AUC={h['auc']:.4f}")
    return idx_active, coef, normH, history

# ══════════════════════════════════════════════════════════════════════════════
# EXPÉRIENCE
# ══════════════════════════════════════════════════════════════════════════════
X_train, y_train, X_test, y_test, X_unlab, y_unlab = load_mnist_binary(
    params["neg_digits"], params["pos_digits"], params["img_size"],
    params["n_train"], params["n_test"], params["n_unlabeled"], seed=params["seed"])
X_all = np.vstack([X_train, X_unlab])
d = X_train.shape[1]
print(f"digits {params['neg_digits']} vs {params['pos_digits']}, "
      f"{params['img_size']}×{params['img_size']} -> dim {d}")
print(f"  train: {len(X_train)}  test: {len(X_test)}  unlabeled: {len(X_unlab)}")

print("\n" + "=" * 72)
print("BASELINE AMBIANTE (PolynomialFeatures, référence historique)")
print("=" * 72)
res_qp2 = classify_poly_qp(X_train, y_train, X_test, y_test, X_all, 2,
                           params["weights"], params["thres1"], params["thres2"],
                           params["const_pen"], params["qp_margin"], params["n_G"],
                           titre="QP ambiant deg=2")
res_ridge = classify_ridge(X_train, y_train, X_test, y_test, 3, titre="Ridge deg=3")

print("\n" + "=" * 72)
print(f"GLOUTON PAR MOMENTS, DEGRÉ FIXE={params['new_deg']} : {params['n_rounds']} tours")
print("=" * 72)
poly0 = PolynomialFeatures(degree=params["deg_start"], include_bias=False)
poly0.fit(np.zeros((1, d)))
powers0 = poly0.powers_
res0 = classify_poly_qp(X_train, y_train, X_test, y_test, X_all, params["deg_start"],
                        params["weights"], params["thres1"], params["thres2"],
                        params["const_pen"], params["qp_margin"], params["n_G"],
                        titre=f"QP deg<={params['deg_start']} (point de départ glouton)")

# ── POOL MAÎTRE : actifs de départ (deg<=deg_start) ∪ TOUT le dictionnaire de
#    degré new_deg. Gram calculé UNE SEULE FOIS ici ; tout le glouton qui suit
#    (screening, sélection, refit à chaque tour) n'est plus que du slicing. ──
full_combos_new_deg = list(combinations_with_replacement(range(d), params["new_deg"]))
cand_powers_all = np.zeros((len(full_combos_new_deg), d), dtype=int)
for i, combo in enumerate(full_combos_new_deg):
    for a in combo: cand_powers_all[i, a] += 1
powers_master = np.vstack([powers0, cand_powers_all])
idx_active_start = list(range(powers0.shape[0]))
idx_cand_all = list(range(powers0.shape[0], powers0.shape[0] + cand_powers_all.shape[0]))
print(f"\nPool maître : {powers_master.shape[0]} monômes "
      f"({powers0.shape[0]} actifs deg<={params['deg_start']} + "
      f"{cand_powers_all.shape[0]} candidats deg={params['new_deg']})")

lo0 = X_all.min(axis=0); hi0 = X_all.max(axis=0)
span0 = np.where(hi0 - lo0 > 1e-12, hi0 - lo0, 1.0)
SC0 = 2.0 / span0
rng_g0 = np.random.default_rng(params["seeds"][43])
idx_g0 = rng_g0.choice(len(X_all), min(params["n_G"], len(X_all)), replace=False)
U_G0 = 2 * (X_all[idx_g0] - lo0) / span0 - 1.0

orders_needed = {o for o in [0, 1, 2, 3]
                 if params["weights"].get(o, 0.) or params["weights_screen"].get(o, 0.)}
print(f"Construction du Gram maître pour ordres {sorted(orders_needed)} "
      f"(économie automatique si w0=0 : ordre 0 jamais calculé)...")
master, mean0 = build_master_grams(U_G0, powers_master, orders_needed, SC0, len(idx_g0),
                                   batch_size=params["batch_size"])
print("Gram maître prêt — plus aucun accès au nuage à partir d'ici.")

sol0 = fit_qp_from_master(X_train, y_train, X_test, y_test, powers_master,
                          idx_active_start, lo0, span0, master, mean0,
                          params["weights"], params["lambda_G"],
                          params["thres1"], params["thres2"], params["const_pen"],
                          params["qp_margin"],
                          titre=f"QP deg<={params['deg_start']} (via Gram maître, cohérence)")

idx_final, coef_final, normH_final, history = run_greedy_fixed_degree_master(
    X_train, y_train, X_test, y_test, powers_master,
    idx_active_start, idx_cand_all, sol0,
    params["n_rounds"], params["dict_size"], params["n_select"],
    master, mean0, params["weights"], params["weights_screen"], params["lambda_G"],
    params["thres1"], params["thres2"], params["const_pen"], params["qp_margin"],
    lo0, span0)
sol1 = history[-1]   # dernier tour, pour le bilan ci-dessous

print("\n" + "=" * 72)
print("BASELINES RBF")
print("=" * 72)
param_grid_svm = {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.01, 0.1, 1]}
svm = GridSearchCV(SVC(kernel="rbf"), param_grid_svm, cv=5, n_jobs=-1)
svm.fit(X_train, y_train)
f_svm = svm.decision_function(X_test)
acc_svm = float(np.mean(np.sign(f_svm) == np.sign(y_test)))
mse_svm = float(np.mean((f_svm - y_test) ** 2))
try:    auc_svm = roc_auc_score((y_test > 0).astype(int), f_svm)
except Exception: auc_svm = float('nan')
print(f"  [SVM RBF]   best={svm.best_params_} | acc_te={acc_svm:.4f} MSE_te={mse_svm:.4f} AUC={auc_svm:.4f}")

param_grid_kr = {"alpha": [1e-3, 1e-2, 1e-1, 1.0], "gamma": [0.001, 0.01, 0.1, 1]}
kr = GridSearchCV(KernelRidge(kernel="rbf"), param_grid_kr, cv=5, n_jobs=-1)
kr.fit(X_train, y_train)
f_kr = kr.predict(X_test)
acc_kr = float(np.mean(np.sign(f_kr) == np.sign(y_test)))
mse_kr = float(np.mean((f_kr - y_test) ** 2))
try:    auc_kr = roc_auc_score((y_test > 0).astype(int), f_kr)
except Exception: auc_kr = float('nan')
print(f"  [Ridge RBF] best={kr.best_params_} | acc_te={acc_kr:.4f} MSE_te={mse_kr:.4f} AUC={auc_kr:.4f}")



print("\n" + "=" * 72)
print("BILAN")
print("=" * 72)

print(f"  QP ambiant deg=2 : "
      f"acc={res_qp2['acc_te']:.4f}  "
      f"acc_seuil={res_qp2['acc_te_thr']:.4f}  "
      f"thr={res_qp2['thr']:.4f}  "
      f"MSE={res_qp2['mse_te']:.4f}  "
      f"AUC={res_qp2['auc']:.4f}  "
      f"‖u‖_H={res_qp2['normH']:.4f}")

print(f"  Ridge deg=3 : "
      f"acc={res_ridge['acc_te']:.4f}  "
      f"MSE={res_ridge['mse_te']:.4f}  "
      f"AUC={res_ridge['auc']:.4f}")

print(f"  QP deg<={params['deg_start']} (via pool) : "
      f"acc={sol0['acc_te']:.4f}  "
      f"acc_seuil={sol0['acc_te_thr']:.4f}  "
      f"thr={sol0['thr']:.4f}  "
      f"MSE={sol0['mse_te']:.4f}  "
      f"AUC={sol0['auc']:.4f}  "
      f"‖u‖_H={sol0['normH']:.4f}")

print(f"  QP deg<={params['deg_start']} + glouton deg={params['new_deg']} "
      f"({params['n_rounds']} tours) : "
      f"acc={sol1['acc_te']:.4f}  "
      f"acc_seuil={sol1['acc_te_thr']:.4f}  "
      f"thr={sol1['thr']:.4f}  "
      f"MSE={sol1['mse_te']:.4f}  "
      f"AUC={sol1['auc']:.4f}  "
      f"‖u‖_H={sol1['normH']:.4f}")

print(f"  SVM RBF : "
      f"acc={acc_svm:.4f}  "
      f"MSE={mse_svm:.4f}  "
      f"AUC={auc_svm:.4f}")

print(f"  Ridge RBF : "
      f"acc={acc_kr:.4f}  "
      f"MSE={mse_kr:.4f}  "
      f"AUC={auc_kr:.4f}")

digits [3, 5] vs [8], 4×4 -> dim 16
  train: 40  test: 500  unlabeled: 4000

BASELINE AMBIANTE (PolynomialFeatures, référence historique)
  [QP ambiant deg=2] deg=2 feat=152 rang=152 | faisable=True marge=1.000 ‖u‖_H=1.6125 MSE_te=2.0997 | acc_tr=1.000 acc_te=0.7420 acc_te_seuil=0.7520 (thr=-0.0133) AUC=0.8203
  [Ridge deg=3] Ridge deg=3 : acc_te=0.7000 MSE_te=10.4600 AUC=0.7550

GLOUTON PAR MOMENTS, DEGRÉ FIXE=4 : 2 tours
  [QP deg<=3 (point de départ glouton)] deg=3 feat=968 rang=865 | faisable=True marge=1.000 ‖u‖_H=0.9839 MSE_te=0.8424 | acc_tr=1.000 acc_te=0.7740 acc_te_seuil=0.7760 (thr=0.0195) AUC=0.8429

Pool maître : 4844 monômes (968 actifs deg<=3 + 3876 candidats deg=4)
Construction du Gram maître pour ordres [1, 2, 3] (économie automatique si w0=0 : ordre 0 jamais calculé)...
Gram maître prêt — plus aucun accès au nuage à partir d'ici.
  [QP deg<=3 (via Gram maître, cohérence)] n_feat=968 rang=956 | faisable=True marge=1.000 ‖u‖_H=1.3472 MSE_te=1.0522 | acc_tr=1.000 acc_te=